In [ ]:
from collections.abc import Callable
from typing import Any, Literal

import equinox as eqx
import jax
import xarray as xr
from context_flux_no.simulations.pde import CubicFlux1D
from context_flux_no.waveforms.fourier import TruncatedFourier1D
from jaxtyping import Array, PRNGKeyArray
from tqdm import tqdm


jax.config.update("jax_enable_x64", True)


In [2]:
def sample_pde(
    pde_factory: Callable[[Any], eqx.Module],
    *,
    seed: int = 0,
    **kwargs: dict[str, tuple[float, float]],
) -> eqx.Module:
    keys = jax.random.split(jax.random.key(seed), len(kwargs))

    param_dict = dict()
    for rng_key, (param_name, sample_range) in zip(keys, kwargs.items()):
        param_dict[param_name] = float(
            jax.random.uniform(
                rng_key,
                minval=sample_range[0],
                maxval=sample_range[1],
            )
        )

    return pde_factory(**param_dict)

In [3]:
pde = sample_pde(CubicFlux1D, a=(-1.0, 1.0), b=(-1.0, 1.0), c=(-1.0, 1.0))
pde

INFO:2025-10-02 21:40:16,440:jax._src.xla_bridge:749: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


2025-10-02 21:40:16,440 INFO CLAW: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


CubicFlux1D(a=0.9398959959797777, b=-0.8387512666713679, c=0.30384451116312405)

In [1]:
from context_flux_no.waveforms.grf import GaussianRandomField1D


u0_func = lambda x_: GaussianRandomField1D.sample(x_, jax.random.key(0))

u, t, x = pde.solve_pyclaw(u0_func, (-1.0, 1.0), 256, (0.0, 0.1), 100, "periodic")

NameError: name 'pde' is not defined

In [ ]:
def sample_coefficients_uniform(
    key: PRNGKeyArray, coeff_range_dict: dict[str, tuple[float, float]]
):
    subkeys = jax.random.split(key, len(coeff_range_dict))
    return {
        name: float(
            jax.random.uniform(subkey, minval=coeff_range[0], maxval=coeff_range[1])
        )
        for subkey, (name, coeff_range) in zip(subkeys, coeff_range_dict.items())
    }


In [8]:
def generate_dataset(
    n_samples: int,
    pde_factory: Callable[[Any], eqx.Module],
    initial_condition_fn: Callable[[Array, PRNGKeyArray], Array],
    coeff_range_dict: dict,
    x_span: tuple[float, float],
    Nx: int,
    t_span: tuple[float, float],
    Nt: int,
    bc: Literal["periodic"] = "periodic",
    seed: int = 0,
):
    keys = jax.random.split(jax.random.key(seed), n_samples)

    solutions = []
    for key in tqdm(keys):
        key_coeff, key_ic = jax.random.split(key)
        coeffs = sample_coefficients_uniform(key_coeff, coeff_range_dict)
        pde = pde_factory(**coeffs)

        sol = pde.solve_pyclaw(
            lambda u0: initial_condition_fn(u0, key_ic), x_span, Nx, t_span, Nt, bc
        )
        solutions.append(*sol, pde.coeffs)

    return xr.concat(solutions, "sample")

In [ ]:
dataset = generate_dataset(
    n_samples=10,
    pde_factory=CubicFlux1D,
    initial_condition_fn=lambda u0, key: TruncatedFourier1D.with_uniform_rand_coeffs(
        num_modes=4, key=key
    )(u0),
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_span=(-1, 1),
    Nx=256,
    t_span=(0, 0.1),
    Nt=100,
    seed=0,
)

  0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_4179890/1382419846.py:19: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  pde = pde_factory(**coeffs)


2025-09-17 16:39:55,859 INFO CLAW: Solution 0 computed for time t=0.000000
2025-09-17 16:40:10,560 INFO CLAW: Solution 1 computed for time t=0.001000
2025-09-17 16:40:35,076 INFO CLAW: Solution 2 computed for time t=0.001929
2025-09-17 16:40:58,219 INFO CLAW: Solution 3 computed for time t=0.001929
